# 00 · Preparar o ambiente e carregar a base em SQL
[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jessicapmartins/segmentacao-clientes-olist/blob/main/notebooks/00_setup_carga_sql.ipynb)

**O que este notebook faz:** conecta ao seu Google Drive, baixa as 9 tabelas públicas da Olist do GitHub oficial e grava tudo em um banco **SQLite** (`olist.db`) na pasta `dados` do projeto.

Rode uma única vez. Os outros notebooks leem o banco direto do Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, sqlite3, pandas as pd

BASE  = '/content/drive/MyDrive/03_Portfolio/Projetos_ML/01_segmentacao-clientes-olist'
DADOS = f'{BASE}/dados'
os.makedirs(DADOS, exist_ok=True)

URL = 'https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/'
ARQUIVOS = {
    'clientes':        'olist_customers_dataset.csv',
    'pedidos':         'olist_orders_dataset.csv',
    'itens_pedido':    'olist_order_items_dataset.csv',
    'pagamentos':      'olist_order_payments_dataset.csv',
    'avaliacoes':      'olist_order_reviews_dataset.csv',
    'produtos':        'olist_products_dataset.csv',
    'vendedores':      'olist_sellers_dataset.csv',
    'categorias_en':   'product_category_name_translation.csv',
    # 'geolocalizacao': 'olist_geolocation_dataset.csv',  # 61 MB: descomente se for usar mapas
}

In [3]:
con = sqlite3.connect(f'{DADOS}/olist.db')
for tabela, arquivo in ARQUIVOS.items():
    df = pd.read_csv(URL + arquivo)
    df.to_sql(tabela, con, if_exists='replace', index=False)
    print(f'{tabela:<16} {len(df):>8,} linhas · {df.shape[1]} colunas')

clientes           99,441 linhas · 5 colunas
pedidos            99,441 linhas · 8 colunas
itens_pedido      112,650 linhas · 7 colunas
pagamentos        103,886 linhas · 5 colunas
avaliacoes         99,224 linhas · 7 colunas
produtos           32,951 linhas · 9 colunas
vendedores          3,095 linhas · 4 colunas
categorias_en          71 linhas · 2 colunas


### Índices: deixam os JOINs mais rápidos

In [4]:
for sql in [
    'CREATE INDEX IF NOT EXISTS ix_ped_cli ON pedidos(customer_id)',
    'CREATE INDEX IF NOT EXISTS ix_itens_ped ON itens_pedido(order_id)',
    'CREATE INDEX IF NOT EXISTS ix_pag_ped ON pagamentos(order_id)',
    'CREATE INDEX IF NOT EXISTS ix_aval_ped ON avaliacoes(order_id)',
    'CREATE INDEX IF NOT EXISTS ix_cli_unico ON clientes(customer_unique_id)',
]:
    con.execute(sql)
con.commit()
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)

,name
0,clientes
1,pedidos
2,itens_pedido
3,pagamentos
4,avaliacoes
5,produtos
6,vendedores
7,categorias_en


### Conferência
Esperado: 99.441 clientes, 99.441 pedidos, 112.650 itens, 103.886 pagamentos, 99.224 avaliações, 32.951 produtos, 3.095 vendedores e 71 categorias.